# Train model

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import re
import warnings
warnings.filterwarnings('ignore') # Tắt các cảnh báo đỏ để log sạch đẹp

# 1. NẠP DỮ LIỆU
X_train = pd.read_csv('X_train.csv')
X_test = pd.read_csv('X_test.csv')
y_train = pd.read_csv('y_train.csv').values.ravel()
y_test = pd.read_csv('y_test.csv').values.ravel()

# Xóa ký tự lỗi cho tên cột
regex = re.compile(r"\[|\]|<", re.IGNORECASE)
X_train.columns = [regex.sub("_", col) if any(x in str(col) for x in set('[ ] <')) else col for col in X_train.columns]
X_test.columns = [regex.sub("_", col) if any(x in str(col) for x in set('[ ] <')) else col for col in X_test.columns]

# ========================================================
# BƯỚC NÉN LOGARIT (GIỮ NGUYÊN VÌ RẤT QUAN TRỌNG)
# ========================================================
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 2. THIẾT LẬP AUTO-TUNING (MỞ RỘNG TỐI ĐA ĐỂ TĂNG ĐỘ CHÍNH XÁC)
param_grid = {
    'n_estimators': [300, 500, 800, 1000, 1500],        # Tăng số cây lên cực lớn
    'max_depth': [4, 6, 8, 10],                         # Cây sâu hơn để học phức tạp hơn
    'learning_rate': [0.01, 0.03, 0.05, 0.1],           # Học chậm mà chắc
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],             # Tỉ lệ mẫu
    'colsample_bytree': [0.5, 0.6, 0.8, 1.0],           # Lấy mẫu cột
    'min_child_weight': [1, 3, 5, 7],                   # [MỚI] Ép phải có đủ dữ liệu mới tách nhánh
    'gamma': [0, 0.1, 0.5, 1, 5]                        # [MỚI] Hình phạt cắt tỉa nhánh vô dụng
}

xgb_model = xgb.XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=150,               # [CỰC QUAN TRỌNG] Tăng lên thử nghiệm 150 tổ hợp tham số khác nhau
    scoring='neg_mean_absolute_error',
    cv=5,                     # [CỰC QUAN TRỌNG] Đánh giá chéo 5 phần chặt chẽ hơn
    verbose=2,                # Hiện tiến trình để bạn không tưởng máy bị treo
    random_state=42
)

print("Đang bật chế độ huấn luyện Vét cạn (Sẽ mất khá nhiều thời gian, vui lòng đợi...).")
random_search.fit(X_train, y_train_log)

best_xgb = random_search.best_estimator_

# 3. DỰ ĐOÁN VÀ ĐÁNH GIÁ
y_pred_log = best_xgb.predict(X_test)
y_pred_real = np.expm1(y_pred_log)

mae = mean_absolute_error(y_test, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_real))
r2 = r2_score(y_test, y_pred_real)

print("\n============================================================")
print("      BÁO CÁO KẾT QUẢ HUẤN LUYỆN MÔ HÌNH XGBOOST (PRO)")
print("============================================================")

print("\n1. THÔNG SỐ TỐI ƯU (BEST PARAMETERS TÌM ĐƯỢC):")
best_params = random_search.best_params_
for param, value in best_params.items():
    print(f"   - {param}: {value}")

print("\n2. ĐÁNH GIÁ ĐỘ CHÍNH XÁC (EVALUATION METRICS):")
print(f"   - MAE (Sai số tuyệt đối)  : {mae:.4f} (Trung bình dự đoán lệch ~{mae:.2f} đơn vị Yield)")
print(f"   - RMSE (Căn bậc hai MSE)  : {rmse:.4f}")
print(f"   - R-squared (R2 Score)    : {r2:.4f} (Giải thích được {r2*100:.2f}% sự biến thiên của dữ liệu)")

print("\n3. TOP 10 YẾU TỐ ẢNH HƯỞNG NHẤT ĐẾN NĂNG SUẤT:")
importances = best_xgb.feature_importances_
feature_names = X_train.columns
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False).head(10)
for index, row in feat_imp.iterrows():
    print(f"   - {row['Feature']:<30}: {row['Importance']*100:.2f}%")

print("\n============================================================")
joblib.dump(best_xgb, 'xgboost_trained_model_MAX.pkl')
print(f"✅ Đã đóng gói và lưu mô hình siêu cấp thành công: 'xgboost_trained_model_MAX.pkl'")

Đang bật chế độ huấn luyện Vét cạn (Sẽ mất khá nhiều thời gian, vui lòng đợi...).
Fitting 5 folds for each of 150 candidates, totalling 750 fits
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=   0.3s
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.8, gamma=5, learning_rate=0.1, max_depth=4, min_child_weight=5, n_estimators=300, subsample=1.0; total time=   0.1s
[CV] END colsample_bytree=0.6, gamma=5, learning_rate=0.1, max_depth=6, min_child_weight=7, n_estimators=1500, subsample=0.6; total tim

KeyboardInterrupt: 

: 